# 03 — OCR: CRNN + CTC

Sequence OCR with CNN feature extractor + BiLSTM and CTC loss.
Three experiments: um_bbox, wm_polygon, wm_bbox.
Compares polygon-path vs bbox-path quality on WM.

In [ ]:
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        from google.colab import drive
        IN_COLAB = True
    except ImportError:
        pass

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    token = userdata.get("GITHUB_TOKEN", "")
    base = f"https://{token}@github.com" if token else "https://github.com"
    if not Path("/content/WaterMeterCV").exists():
        subprocess.run(["git", "clone", f"{base}/UrranQx/WaterMeterCV.git", "/content/WaterMeterCV"], check=True)
    BRANCH = "feature/ocr-notebooks"
    subprocess.run(["git", "-C", "/content/WaterMeterCV", "checkout", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "rapidfuzz", "shapely"], check=True)
    ROOT         = Path("/content/WaterMeterCV")
    DATA_ROOT    = Path("/content/drive/MyDrive/WaterMeterCV/WaterMetricsDATA")
    WEIGHTS_BASE = Path("/content/drive/MyDrive/WaterMeterCV/weights")
    RESULTS_DIR  = Path("/content/drive/MyDrive/WaterMeterCV/results")
    WORKERS = 2
else:
    ROOT         = Path("../..")
    DATA_ROOT    = ROOT / "WaterMetricsDATA"
    WEIGHTS_BASE = ROOT / "models/weights"
    RESULTS_DIR  = ROOT / "results"
    WORKERS = 0

sys.path.insert(0, str(ROOT))
WEIGHTS_BASE.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import json, time, csv
import torch, cv2, numpy as np
from datetime import datetime

print(f"ROOT: {ROOT}")
print(f"torch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
CROPS_ROOT   = DATA_ROOT / "ocr_crops"
WM_PATH      = DATA_ROOT / "waterMeterDataset/WaterMeters"
UM_YOLO_PATH = DATA_ROOT / "utility-meter-reading-dataset-for-automatic-reading-yolo.v4i.yolov11"
WEIGHTS_DIR  = WEIGHTS_BASE / "ocr_crnn_ctc"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_H, IMG_W = 64, 256
EPOCHS      = 30
BATCH_SIZE  = 32
LR          = 1e-3
NUM_CLASSES = 11   # 10 digits + CTC blank (index 10)
RUN_NAME    = f"crnn_ctc_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
print(f"Device: {DEVICE}")
print(f"Run: {RUN_NAME}")

In [ ]:
import torch.nn as nn
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from PIL import Image

from models.data.ocr_dataset import prepare_ocr_crops, load_ocr_crops, CHARSET
from models.metrics.evaluation import full_string_accuracy, per_digit_accuracy, character_error_rate

prepare_ocr_crops(WM_PATH, UM_YOLO_PATH, CROPS_ROOT)
print("Crops ready.")

In [ ]:
class CRNNCTC(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES, hidden: int = 256):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d((2, 1)),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d((2, 1)),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, None))
        self.rnn  = nn.LSTM(256, hidden, num_layers=2, bidirectional=True, batch_first=False)
        self.fc   = nn.Linear(hidden * 2, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = self.pool(x).squeeze(2).permute(2, 0, 1)  # (T, B, 256)
        x, _ = self.rnn(x)                              # (T, B, 2*hidden)
        return self.fc(x).log_softmax(2)                # (T, B, C)


class CropDataset(Dataset):
    def __init__(self, samples: list):  # [(img_path, label_str)]
        self.samples = samples

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        img_path, label_str = self.samples[idx]
        img    = cv2.imread(str(img_path))
        tensor = TF.to_tensor(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        label  = torch.tensor([CHARSET.index(c) for c in label_str if c in CHARSET], dtype=torch.long)
        return tensor, label


def collate_ctc(batch):
    imgs, labels = zip(*batch)
    lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    return torch.stack(imgs), torch.cat(labels), lengths


def ctc_decode(log_probs: torch.Tensor) -> list:
    preds = log_probs.argmax(2).permute(1, 0).cpu().numpy()
    results = []
    for seq in preds:
        chars, prev = [], -1
        for c in seq:
            if c != prev and c != 10:
                chars.append(CHARSET[c])
            prev = c
        results.append("".join(chars))
    return results


def orient_fix(pred: str) -> str:
    rev = pred[::-1]
    return rev if rev.isdigit() else pred


def train_ctc(model, loader, epochs, device, best_path=None):
    model.to(device)
    opt       = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.StepLR(opt, step_size=max(1, epochs//2), gamma=0.1)
    criterion = nn.CTCLoss(blank=10, zero_infinity=True)
    best_loss = float("inf")
    for epoch in range(epochs):
        model.train(); total = 0.0
        for imgs, labels, lengths in tqdm(loader, desc=f"epoch {epoch+1}/{epochs}", leave=False):
            imgs, labels, lengths = imgs.to(device), labels.to(device), lengths.to(device)
            log_p   = model(imgs)
            inp_len = torch.full((imgs.size(0),), log_p.size(0), dtype=torch.long, device=device)
            loss = criterion(log_p, labels, inp_len, lengths)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item()
        scheduler.step()
        avg = total / len(loader)
        if best_path and avg < best_loss:
            best_loss = avg; torch.save(model.state_dict(), best_path)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  epoch {epoch+1}/{epochs}  loss={avg:.4f}")
    return model


@torch.no_grad()
def eval_ctc(model, samples, device):
    model.eval()
    preds, gts, times = [], [], []
    for img_path, label_gt in samples:
        img    = cv2.imread(str(img_path))
        tensor = TF.to_tensor(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)).unsqueeze(0).to(device)
        t0     = time.perf_counter()
        log_p  = model(tensor)
        times.append((time.perf_counter() - t0) * 1000)
        pred = orient_fix(ctc_decode(log_p)[0])
        preds.append(pred); gts.append(label_gt)
    fsa_raw  = full_string_accuracy(preds, gts)
    fsa_norm = full_string_accuracy([p.lstrip("0") or "0" for p in preds],
                                    [g.lstrip("0") or "0" for g in gts])
    pda = float(np.mean([per_digit_accuracy(p, g) for p, g in zip(preds, gts)]))
    cer = float(np.mean([character_error_rate(p, g) for p, g in zip(preds, gts)]))
    ms  = float(np.mean(times))
    return fsa_raw, fsa_norm, pda, cer, ms, preds, gts


print("Helpers loaded.")

In [ ]:
um_train = load_ocr_crops(CROPS_ROOT / "um_bbox", "train")
um_test  = load_ocr_crops(CROPS_ROOT / "um_bbox", "test")
um_loader = DataLoader(CropDataset(um_train), batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=WORKERS, collate_fn=collate_ctc)
print(f"UM train={len(um_train)}, test={len(um_test)}")
model_um = CRNNCTC()
model_um = train_ctc(model_um, um_loader, EPOCHS, DEVICE, WEIGHTS_DIR / f"um_best_{RUN_NAME}.pth")
torch.save(model_um.state_dict(), WEIGHTS_DIR / f"um_last_{RUN_NAME}.pth")
um_fsa_raw, um_fsa_norm, um_pda, um_cer, um_ms, um_preds, um_gts = eval_ctc(model_um, um_test, DEVICE)
print(f"UM — FSA raw={um_fsa_raw:.3f} norm={um_fsa_norm:.3f} PDA={um_pda:.3f} CER={um_cer:.3f} {um_ms:.1f}ms")

In [ ]:
wm_poly_train = load_ocr_crops(CROPS_ROOT / "wm_polygon", "train")
wm_poly_test  = load_ocr_crops(CROPS_ROOT / "wm_polygon", "test")
wm_poly_loader = DataLoader(CropDataset(wm_poly_train), batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=WORKERS, collate_fn=collate_ctc)
print(f"WM polygon train={len(wm_poly_train)}, test={len(wm_poly_test)}")
model_wm_poly = CRNNCTC()
model_wm_poly = train_ctc(model_wm_poly, wm_poly_loader, EPOCHS, DEVICE,
                           WEIGHTS_DIR / f"wm_poly_best_{RUN_NAME}.pth")
torch.save(model_wm_poly.state_dict(), WEIGHTS_DIR / f"wm_poly_last_{RUN_NAME}.pth")
wm_poly_fsa_raw, wm_poly_fsa_norm, wm_poly_pda, wm_poly_cer, wm_poly_ms, wm_poly_preds, wm_poly_gts = eval_ctc(model_wm_poly, wm_poly_test, DEVICE)
print(f"WM poly — FSA raw={wm_poly_fsa_raw:.3f} norm={wm_poly_fsa_norm:.3f} PDA={wm_poly_pda:.3f} CER={wm_poly_cer:.3f} {wm_poly_ms:.1f}ms")

In [ ]:
wm_bbox_train = load_ocr_crops(CROPS_ROOT / "wm_bbox", "train")
wm_bbox_test  = load_ocr_crops(CROPS_ROOT / "wm_bbox", "test")
wm_bbox_loader = DataLoader(CropDataset(wm_bbox_train), batch_size=BATCH_SIZE, shuffle=True,
                            num_workers=WORKERS, collate_fn=collate_ctc)
print(f"WM bbox train={len(wm_bbox_train)}, test={len(wm_bbox_test)}")
model_wm_bbox = CRNNCTC()
model_wm_bbox = train_ctc(model_wm_bbox, wm_bbox_loader, EPOCHS, DEVICE,
                           WEIGHTS_DIR / f"wm_bbox_best_{RUN_NAME}.pth")
torch.save(model_wm_bbox.state_dict(), WEIGHTS_DIR / f"wm_bbox_last_{RUN_NAME}.pth")
wm_bbox_fsa_raw, wm_bbox_fsa_norm, wm_bbox_pda, wm_bbox_cer, wm_bbox_ms, wm_bbox_preds, wm_bbox_gts = eval_ctc(model_wm_bbox, wm_bbox_test, DEVICE)
print(f"WM bbox — FSA raw={wm_bbox_fsa_raw:.3f} norm={wm_bbox_fsa_norm:.3f} PDA={wm_bbox_pda:.3f} CER={wm_bbox_cer:.3f} {wm_bbox_ms:.1f}ms")

In [ ]:
print(f"{'='*70}")
print(f"{'Metric':<20} {'UM bbox':>12} {'WM polygon':>12} {'WM bbox':>12}")
print(f"{'='*70}")
for name, um_v, poly_v, bbox_v in [
    ("FSA raw",   um_fsa_raw,  wm_poly_fsa_raw,  wm_bbox_fsa_raw),
    ("FSA norm",  um_fsa_norm, wm_poly_fsa_norm, wm_bbox_fsa_norm),
    ("Per-digit", um_pda,      wm_poly_pda,      wm_bbox_pda),
    ("CER",       um_cer,      wm_poly_cer,      wm_bbox_cer),
]:
    print(f"{name:<20} {um_v:>12.3f} {poly_v:>12.3f} {bbox_v:>12.3f}")
print(f"{'Inference ms':<20} {um_ms:>12.1f} {wm_poly_ms:>12.1f} {wm_bbox_ms:>12.1f}")
print(f"{'='*70}")

In [ ]:
# Visualization: 3x4 grid — UM, WM polygon, WM bbox rows
fig, axes = plt.subplots(3, 4, figsize=(20, 10))
rows = [
    ("UM bbox",    CROPS_ROOT / "um_bbox" / "test",    um_preds,      um_gts),
    ("WM polygon", CROPS_ROOT / "wm_polygon" / "test", wm_poly_preds, wm_poly_gts),
    ("WM bbox",    CROPS_ROOT / "wm_bbox" / "test",    wm_bbox_preds, wm_bbox_gts),
]
for row_i, (label, crops_dir, preds, gts) in enumerate(rows):
    samples = load_ocr_crops(crops_dir.parent, "test")
    for col_i, ax in enumerate(axes[row_i]):
        if col_i >= len(samples):
            ax.axis("off"); continue
        img = cv2.imread(str(samples[col_i][0]))
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        gt  = gts[col_i] if col_i < len(gts) else "?"
        pr  = preds[col_i] if col_i < len(preds) else "?"
        ax.set_title(f"GT={gt} Pred={pr}", fontsize=9)
        ax.axis("off")
    axes[row_i][0].set_ylabel(label, fontsize=11)
plt.suptitle("CRNN+CTC Predictions", fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ocr_crnn_ctc_predictions.png", dpi=150)
plt.close()
print("Saved ocr_crnn_ctc_predictions.png")

In [ ]:
metrics = {
    "method": "crnn_ctc",
    "utility_meter":  {"fsa_raw": round(um_fsa_raw,4), "fsa_norm": round(um_fsa_norm,4),
                       "per_digit_acc": round(um_pda,4), "cer": round(um_cer,4),
                       "avg_inference_ms": round(um_ms,1), "n_test": len(um_gts)},
    "wm_polygon":     {"fsa_raw": round(wm_poly_fsa_raw,4), "fsa_norm": round(wm_poly_fsa_norm,4),
                       "per_digit_acc": round(wm_poly_pda,4), "cer": round(wm_poly_cer,4),
                       "avg_inference_ms": round(wm_poly_ms,1)},
    "wm_bbox":        {"fsa_raw": round(wm_bbox_fsa_raw,4), "fsa_norm": round(wm_bbox_fsa_norm,4),
                       "per_digit_acc": round(wm_bbox_pda,4), "cer": round(wm_bbox_cer,4),
                       "avg_inference_ms": round(wm_bbox_ms,1)},
    "config": {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LR, "architecture": "CRNN+CTC", "lstm_hidden": 256},
    "run_date": datetime.now().isoformat(),
}
with open(RESULTS_DIR / "ocr_crnn_ctc_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

csv_path = RESULTS_DIR / "ocr_comparison.csv"
header = ["method","um_fsa_raw","um_fsa_norm","um_pda","um_cer","um_ms",
          "wm_poly_fsa_raw","wm_poly_fsa_norm","wm_poly_pda","wm_poly_cer","wm_poly_ms",
          "wm_bbox_fsa_raw","wm_bbox_fsa_norm","wm_bbox_pda","wm_bbox_cer","wm_bbox_ms","run_date"]
write_hdr = not csv_path.exists()
with open(csv_path, "a", newline="") as f:
    w = csv.writer(f)
    if write_hdr: w.writerow(header)
    w.writerow(["crnn_ctc",
                round(um_fsa_raw,4), round(um_fsa_norm,4), round(um_pda,4), round(um_cer,4), round(um_ms,1),
                round(wm_poly_fsa_raw,4), round(wm_poly_fsa_norm,4), round(wm_poly_pda,4), round(wm_poly_cer,4), round(wm_poly_ms,1),
                round(wm_bbox_fsa_raw,4), round(wm_bbox_fsa_norm,4), round(wm_bbox_pda,4), round(wm_bbox_cer,4), round(wm_bbox_ms,1),
                datetime.now().strftime("%Y-%m-%d %H:%M")])
print("Results saved.")

## Conclusions

- **UM:** FSA raw=?, FSA norm=?, PDA=?, CER=?, ?ms
- **WM polygon:** FSA raw=?, FSA norm=?, PDA=?, CER=?, ?ms
- **WM bbox:** FSA raw=?, FSA norm=?, PDA=?, CER=?, ?ms
- **CRNN vs CNN:** fill after running — BiLSTM adds sequence context.
- **Key finding:** fill after running.